# FABQ-RC: Evaluate Quantized Mistral-7B on Paperspace

**Runtime:** Paperspace RTX 5000 (16GB VRAM) · ~30-45 min
**Purpose:** Evaluate FABQ-RC quantized Mistral-7B perplexity on WikiText-2 and downstream benchmarks

**Note:** This notebook loads a fresh FP16 Mistral-7B and applies the FABQ-RC quantized weights.

## 0. Setup

In [ ]:
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate datasets
!pip install -q pandas numpy==1.26.4 tqdm matplotlib seaborn

import os, math, json, time, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Load FABQ-RC Quantized State Dict

In [ ]:
# Find available quantized weight file
QUANTIZED_PATH = None
for pattern in ["quantized_mistral7b_fabqrc*.pth", "quantized_mistral7b_fabqrc.pth"]:
    files = sorted(glob.glob(pattern), key=lambda x: (len(x), x))
    if files:
        QUANTIZED_PATH = files[-1]
        break

if QUANTIZED_PATH is None:
    raise FileNotFoundError(
        f"No quantized weight file found. Available: {glob.glob('*.pth')}"
    )

print(f"Loading quantized state dict from {QUANTIZED_PATH}...")
t0 = time.time()
quantized_state = torch.load(QUANTIZED_PATH, map_location='cpu', weights_only=False)
print(f"Loaded in {time.time()-t0:.1f}s")
print(f"Total keys: {len(quantized_state)}")

# Identify quantized layers
layer_names = set()
for key in quantized_state.keys():
    if '.int8_channels' in key:
        layer_names.add(key.replace('.int8_channels', ''))
print(f"Quantized layers: {len(layer_names)}")

## 2. Apply FABQ-RC Weights to Mistral-7B

In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-v0.1"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

print("Loading FP16 Mistral-7B...")
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.eval()
print(f"Loaded in {time.time()-t0:.1f}s")
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")

In [ ]:
def apply_fabqrc_weights(model, state_dict):
    """Replace model weights with FABQ-RC quantized weights."""
    layers_applied = 0
    skipped = 0
    
    for name, module in model.named_modules():
        if not isinstance(module, nn.Linear):
            continue
        
        layer_name = name
        int8_channels_key = f'{layer_name}.int8_channels'
        
        if int8_channels_key not in state_dict:
            continue
        
        try:
            int8_channels = state_dict[f'{layer_name}.int8_channels']
            binary_channels = state_dict[f'{layer_name}.binary_channels']
            int8_weights = state_dict[f'{layer_name}.int8_weights']
            int8_scales = state_dict[f'{layer_name}.int8_scales']
            binary_recon = state_dict[f'{layer_name}.binary_reconstructed_weights']
            bias = state_dict.get(f'{layer_name}.bias', None)
            
            n_int8 = len(int8_channels)
            n_binary = len(binary_channels)
            in_feat = int8_weights.shape[1] if n_int8 > 0 else binary_recon.shape[1]
            out_feat = n_int8 + n_binary
            
            weight = torch.zeros((out_feat, in_feat), dtype=torch.float16)
            
            if n_int8 > 0:
                weight[int8_channels] = int8_weights.float() * int8_scales.unsqueeze(-1)
            
            if n_binary > 0:
                weight[binary_channels] = binary_recon.float()
            
            with torch.no_grad():
                module.weight.copy_(weight.to(module.weight.device))
                if bias is not None:
                    module.bias.copy_(bias.to(module.bias.device))
            
            layers_applied += 1
        except Exception as e:
            print(f"Warning: Failed to apply weights to {layer_name}: {e}")
            skipped += 1
    
    print(f"Applied FABQ-RC weights to {layers_applied} layers, skipped {skipped}")
    return layers_applied

print("Applying FABQ-RC weights...")
t0 = time.time()
layers_applied = apply_fabqrc_weights(model, quantized_state)
print(f"Weight application complete in {time.time()-t0:.1f}s")

## 3. Compute Bits Per Parameter

In [ ]:
total_bits, total_params = 0, 0
n_int8_total, n_binary_total = 0, 0

for layer_name in layer_names:
    int8_channels = quantized_state[f'{layer_name}.int8_channels']
    binary_channels = quantized_state[f'{layer_name}.binary_channels']
    int8_weights = quantized_state[f'{layer_name}.int8_weights']
    binary_recon = quantized_state[f'{layer_name}.binary_reconstructed_weights']
    
    n_int8 = len(int8_channels)
    n_binary = len(binary_channels)
    in_feat = int8_weights.shape[1] if n_int8 > 0 else binary_recon.shape[1]
    out_feat = n_int8 + n_binary
    n = out_feat * in_feat
    
    total_params += n
    total_bits += n_int8 * in_feat * 8 + n_binary * in_feat * 1
    n_int8_total += n_int8
    n_binary_total += n_binary

bpw = total_bits / total_params if total_params > 0 else 0
print(f"FABQ-RC: {bpw:.4f} bits/parameter")
print(f"  int8 channels: {n_int8_total:,}")
print(f"  binary channels: {n_binary_total:,}")
print(f"  Total params: {total_params:,}")

## 4. Perplexity Evaluation on WikiText-2

In [ ]:
from datasets import load_dataset

def compute_perplexity(model, texts, tokenizer, device, stride=512, max_samples=512):
    """Compute perplexity on a list of texts. Lower is better."""
    model.eval()
    total_loss, total_tokens = 0.0, 0
    
    for i, text in enumerate(texts[:max_samples]):
        if not text or len(text.strip()) < 20:
            continue
        enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=stride)
        input_ids = enc['input_ids'].to(device)
        if input_ids.numel() < 20:
            continue
        
        with torch.no_grad():
            outputs = model(input_ids)
            logits = outputs.logits
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = input_ids[..., 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1), reduction='mean'
            )
        total_loss += loss.item() * shift_labels.numel()
        total_tokens += shift_labels.numel()
    
    return math.exp(total_loss / total_tokens) if total_tokens > 0 else float('inf')

print("Loading WikiText-2 test set...")
wikitext = load_dataset("wikitext", "wikitext-2-v1", split="test")
wikitext_texts = [t for t in wikitext['text'] if t.strip() and len(t.strip()) > 20]
print(f"WikiText-2: {len(wikitext_texts)} valid texts")

In [ ]:
print("Evaluating FABQ-RC perplexity...")
t0 = time.time()
ppl_fabqrc = compute_perplexity(model, wikitext_texts, tokenizer, DEVICE, max_samples=512)
print(f"FABQ-RC perplexity: {ppl_fabqrc:.4f} ({time.time()-t0:.1f}s)")

## 5. Compare with FP16 Baseline

In [ ]:
import gc

# Free up VRAM before loading FP16 baseline
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Loading FP16 baseline for comparison...")
t0 = time.time()
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model_fp16.eval()
print(f"FP16 model loaded in {time.time()-t0:.1f}s")

print("Evaluating FP16 baseline perplexity...")
t0 = time.time()
ppl_fp16 = compute_perplexity(model_fp16, wikitext_texts, tokenizer, DEVICE, max_samples=512)
print(f"FP16 perplexity: {ppl_fp16:.4f} ({time.time()-t0:.1f}s)")

overhead = (ppl_fabqrc / ppl_fp16 - 1) * 100 if ppl_fp16 > 0 else 0
print(f"\nOverhead: {overhead:+.2f}%")

## 6. Summary

In [ ]:
print("="*60)
print("FABQ-RC QUANTIZED MISTRAL-7B EVALUATION")
print("="*60)
print(f"Model: Mistral-7B-v0.1")
print(f"Method: FABQ-RC (Fisher-Adaptive Binary Quantization)")
print(f"Quantized file: {QUANTIZED_PATH}")
print(f"Bits/parameter: {bpw:.4f}")
print(f"Quantized layers: {len(layer_names)}")
print()
print(f"WikiText-2 Perplexity:")
print(f"  FP16 baseline: {ppl_fp16:.4f}")
print(f"  FABQ-RC: {ppl_fabqrc:.4f} ({overhead:+.2f}% overhead)")
print("="*60)